# Combining ACS 5 Year Metrics and Property Data

This notebook will require you to have ingested block groups (notebooks, 02_ingest, tiles, ingest_tiles), any parcel layer (notebooks, 02_ingest, parcels, ingest_parcels), and the census data (notebooks, 02_ingest, population, ingest_population). 

Here are the census metrics available to you: 

- b01001   # Sex by Age
- b19013   # Median Household Income
- b02001   # Race
- b25077   # Median Property Value (Dollars)
- b17001   # Poverty Status in the Past 12 Months
- b15003   # Educational attainment for over 25yo
- b23025   # Employment status (Labor force, unemployment) 
- b08303   # Travel time to work
- b25002   # Occupancy Status 
- b25064   # Median Gross Rent
- b25003   # Tenure (Owner vs Renter)
- b25091   # Mortgage Status by Selected Monthly Owner Costs
- b25070   # Gross Rent as % of Household Income (Rent Burden)
- b03002   # Hispanic or Latino Origin by Race
- c16002   # Household Language by English Proficiency
- b11004   # Family Type by Presence of Own Children
- b01003   # Total Population
- b22010   # Receipt of Food Stamps/SNAP

## ACS

### Import ACS

In [ ]:
# Get ACS data for selected metrics
from openplaces.api import get_dataset
import pandas as pd
import numpy as np

partitions = [
    'b19013',  # Median Household Income
    'b02001',  # Race
    'b25077',  # Median Property Value (Dollars)
    'b17001',  # Poverty Status
    'b15003',  # Educational attainment
    'b23025',  # Employment status
    'b08303',  # Travel time to work
    'b25064',  # Median Gross Rent
    'b25070',  # Rent Burden
    'b03002',  # Hispanic or Latino Origin by Race
    'b01003',  # Total pop
]

dfs = []

for pid in partitions:
    df = get_dataset(recipe='US_population-acs-2024', partition_id=pid)

    # ensure index is set
    if 'census_geo_id' in df.columns:
        df = df.set_index('census_geo_id')

    dfs.append(df)

acs = pd.concat(dfs, axis=1, join='outer')

### Compute new metrics 

In [ ]:
acs['pct_black'] = acs['race_black'] / acs['total_pop']
acs['pct_hispanic'] = acs['eth_hispanic_or_latino'] / acs['total_pop']
acs['pct_poverty'] = acs['inc_below_pov_level'] / acs['total_pop']

no_college_cols = [
    'ed_no_schooling',
    'ed_nursery',
    'ed_kindergarten',
    'ed_1st_grade',
    'ed_2nd_grade',
    'ed_3rd_grade',
    'ed_4th_grade',
    'ed_5th_grade',
    'ed_6th_grade',
    'ed_7th_grade',
    'ed_8th_grade',
    'ed_9th_grade',
    'ed_10th_grade',
    'ed_11th_grade',
    'ed_12th_grade_no_diploma',
    'ed_hs_diploma',
    'ed_ged_alt_hs_diploma',
    'ed_some_college_lt_1_year',
    'ed_some_college_1_or_more_yrs_no_degree',
]

acs['no_college_total'] = acs[no_college_cols].sum(axis=1)
acs['pct_no_college'] = acs['no_college_total'] / acs['pop_age_25_plus']

acs['lf_participation_rate'] = acs['labor_force'] / acs['pop_age_16_plus']

acs['unemployment_rate'] = (
    acs['civilian_labor_force_unemployed'] / acs['civilian_labor_force']
)

commute_midpoints = {
    'commute_lt_5_min': 2.5,
    'commute_5_to_9_min': 7,
    'commute_10_to_14_min': 12,
    'commute_15_to_19_min': 17,
    'commute_20_to_24_min': 22,
    'commute_25_to_29_min': 27,
    'commute_30_to_34_min': 32,
    'commute_35_to_39_min': 37,
    'commute_40_to_44_min': 42,
    'commute_45_to_59_min': 52,
    'commute_60_to_89_min': 75,
    'commute_90_or_more_min': 100,
}

commute_cols = list(commute_midpoints.keys())

acs['total_commuters'] = acs[commute_cols].sum(axis=1)

acs['avg_commute_time'] = (
    sum(acs[col] * midpoint for col, midpoint in commute_midpoints.items())
    / acs['total_commuters']
)

acs['rent_burdened_households'] = (
    acs['rent_burden_30_to_34']
    + acs['rent_burden_35_to_39']
    + acs['rent_burden_40_to_49']
    + acs['rent_burden_ge_50']
)

acs['pct_rent_burdened'] = acs['rent_burdened_households'] / acs['renter_households']

cols = [
    # original ACS variables
    'med_household_inc',
    'med_home_val',
    'med_gross_rent',
    # computed variables
    'pct_black',
    'pct_hispanic',
    'pct_poverty',
    'pct_no_college',
    'lf_participation_rate',
    'unemployment_rate',
    'avg_commute_time',
    'pct_rent_burdened',
]

acs = acs[cols]
acs.head(1)

### Check for missing values at BG or tract level. 

In [ ]:
tract = acs[acs.index.astype(str).str.startswith('140000')]
bg = acs[acs.index.astype(str).str.startswith('150000')]


def coverage_table(subdf):
    invalid = (subdf < 0) | (subdf.isna())
    pct_missing = invalid.sum(axis=0) / len(subdf)
    return pct_missing


tract_missing = coverage_table(tract)
bg_missing = coverage_table(bg)

coverage = pd.DataFrame(
    {'pct_missing_tract': tract_missing, 'pct_missing_bg': bg_missing}
)


coverage = coverage.sort_values(
    by=['pct_missing_bg', 'pct_missing_tract'], ascending=False
)

coverage = coverage[
    (coverage['pct_missing_tract'] > 0) | (coverage['pct_missing_bg'] > 0)
]

coverage

### Next, we import our block group data, and merge on census_geo_id to the ACS metrics

In [ ]:
# Get block groups
from openplaces.api import get_entities

bg = get_entities(recipe='US_tile-census-2025_blockgroup', geom=True)
ma_bg = bg[bg['admin2_id'] == 'US-MA']

tract = get_entities(recipe='US_tile-census-2025_tract', geom=True)
ma_tract = tract[tract['admin2_id'] == 'US-MA']

In [ ]:
ma_bg_census = ma_bg.merge(acs, on='census_geo_id', how='left')
ma_tract_census = ma_tract.merge(acs, on='census_geo_id', how='left')

# exclude ID + geometry columns from numeric operations
exclude = [
    'census_geo_id',
    'geometry',
    'admin3_id_admin1',
    'admin3_id',
    'admin2_id',
    'geometry',
]
work_cols = ma_bg_census.columns.difference(exclude)

# ensure numeric only where appropriate
ma_bg_census[work_cols] = ma_bg_census[work_cols].apply(pd.to_numeric, errors='coerce')
ma_tract_census[work_cols] = ma_tract_census[work_cols].apply(
    pd.to_numeric, errors='coerce'
)

# remove ACS negative codes only on numeric columns
ma_bg_census[work_cols] = ma_bg_census[work_cols].mask(
    ma_bg_census[work_cols] < 0, np.nan
)
ma_tract_census[work_cols] = ma_tract_census[work_cols].mask(
    ma_tract_census[work_cols] < 0, np.nan
)

# geometry cleaning
ma_bg_census = ma_bg_census[ma_bg_census.geometry.notna()]
ma_tract_census = ma_tract_census[ma_tract_census.geometry.notna()]

ma_bg_census = ma_bg_census[ma_bg_census.geometry.is_valid]
ma_tract_census = ma_tract_census[ma_tract_census.geometry.is_valid]

### Color plot of the metrics we pulled in using matplotlib and geopandas. 

In [ ]:
import matplotlib.pyplot as plt

tract_gdf = ma_tract_census
bg_gdf = ma_bg_census

tract_cols = [
    'pct_poverty',
    'med_gross_rent',
    'med_home_val',
    'med_household_inc',
]

bg_cols = [
    'pct_black',
    'pct_hispanic',
    'pct_no_college',
    'lf_participation_rate',
    'unemployment_rate',
    'avg_commute_time',
    'pct_rent_burdened',
]

all_cols = tract_cols + bg_cols

n = len(all_cols)
fig, axes = plt.subplots(n, 1, figsize=(10, 3 * n))

if n == 1:
    axes = [axes]

for ax, col in zip(axes, all_cols):
    # choose correct dataset
    if col in tract_cols:
        gdf = tract_gdf
        cmap = 'Reds'
    else:
        gdf = bg_gdf
        cmap = 'viridis_r'

    gdf.plot(
        column=col,
        ax=ax,
        legend=True,
        cmap=cmap,
        missing_kwds={'color': 'lightgrey', 'label': 'Missing'},
    )

    ax.set_title(col.replace('_', ' ').title())
    ax.axis('off')

plt.tight_layout()
plt.show()

### Next, we can pull in our Massachusetts properties. 

In [ ]:
from openplaces.api import get_entities
import geopandas as gpd

prop = get_entities(
    recipe='US-MA_parcel-massgis-2025', admin_id='US-MA-NO', layer='property'
)
parcel = get_entities(
    recipe='US-MA_parcel-massgis-2025', admin_id='US-MA-NO', geom=True
)

In [ ]:
parcel_geom = parcel[['parcel_id_admin2', 'geometry']]

# merge onto prop
prop_merged = prop.merge(parcel_geom, on='parcel_id_admin2', how='left')

prop_merged = gpd.GeoDataFrame(prop_merged, geometry='geometry')

### We can use a spatial join to connect our property attributes to census data.

In [ ]:
ma_tract_census = ma_tract_census[tract_cols + ['geometry']].copy()
ma_bg_census = ma_bg_census[bg_cols + ['geometry']].copy()

In [ ]:
prop_census = gpd.sjoin(
    prop_merged, ma_tract_census, how='left', predicate='intersects'
)
prop_census = gpd.sjoin(prop_census, ma_bg_census, how='left', predicate='intersects')

### Narrow down to single-family homes and real sales (above 200k)

In [ ]:
# Narrow down on single family homes

prop_census['last_sale_date'] = pd.to_datetime(
    prop_census['last_sale_date'], errors='coerce'
)

single_fam = prop_census[prop_census['usecode'].isin(['101', '1010'])]

single_fam = single_fam[
    single_fam['last_sale_price'].notna() & (single_fam['last_sale_price'] > 200000)
].copy()

single_fam.head(1)

In [ ]:
import matplotlib.pyplot as plt

ax = parcel.plot(color='lightgrey', figsize=(10, 10), alpha=0.5)

high_value = single_fam[single_fam['med_home_val'] > 700_000]

recent = single_fam[single_fam['last_sale_date'] > '2023-01-01']

affordable = recent[recent['last_sale_price'] < 500_000]

mid_market = recent[
    (recent['last_sale_price'] >= 500_000) & (recent['last_sale_price'] <= 800_000)
]

unaffordable = recent[recent['last_sale_price'] > 800_000]

high_value.plot(ax=ax, color='blue', alpha=0.25)

affordable.plot(ax=ax, color='green', markersize=5)
mid_market.plot(ax=ax, color='orange', markersize=5)
unaffordable.plot(ax=ax, color='red', markersize=5)

plt.title(
    'Single-Family Housing Price Composition in High-Value Neighborhoods (Since 2023)'
)
plt.axis('off')

summary_text = (
    f'Affordable Sales (<$500k): {len(affordable):,}\n'
    f'Mid-market Sales ($500k–$800k): {len(mid_market):,}\n'
    f'Unaffordable Sales (>$800k): {len(unaffordable):,}'
)

plt.figtext(0.5, 0.01, summary_text, ha='center', fontsize=10)
plt.subplots_adjust(bottom=0.15)

plt.show()